In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_sales_orders AS
SELECT
  rso.numero_ov,
  rso.data,
  rso.tipo_ov,
  rso.org_vendas,
  rso.canal_dist,
  rso.emissor_da_ordem,
  rso.centro,
  rso.material,  
  rso.quantidade,
  mc.item_principal_cadeia,
  k.cen AS centro_original
FROM parts_hdbk_sandbox.dt_sales_orders.raw_sales_order rso
LEFT JOIN parts_hdbk_sandbox.pr_cadastrao.material_cadeia mc
  ON rso.material = mc.material
  AND rso.org_vendas = mc.empresa
LEFT JOIN parts_hdbk_sandbox.dm_customers.knvv_sap k
  ON rso.emissor_da_ordem = k.cliente
  AND rso.org_vendas = k.orgv
  AND rso.canal_dist = k.cdst
  AND rso.setor_ativ = k.sa
WHERE rso.data >= '2025-01-01'

In [0]:
def gerar_layout_pivotado(
    df,
    aba,
    data_minima,
    org_vendas,
    mes,
    ano,
    operacao,
    sufixo_arquivo="",
    canal_dist=None,
    centro_original=None,
    tipo_ov=None,
    linha='item_principal_cadeia',  # ou 'material'
    incluir_cliente=False  # True = adiciona emissor_da_ordem como agrupador
):
    # Mapeamento de org_vendas para prefixo do arquivo
    prefixos = {
        "0200": "HDA",
        "0500": "HAB"
    }
    prefixo = prefixos.get(org_vendas, "OUT")

    # Meses abreviados em português
    meses_pt = {
        1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
        5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
        9: "Set", 10: "Out", 11: "Nov", 12: "Dez"
    }
    mes_abrev = meses_pt.get(mes, "")

    # Monta o nome do arquivo: ex. "HDA 2026 Jun Relatorio"
    arquivo = f"{prefixo} {ano} {mes_abrev}"
    if sufixo_arquivo:
        arquivo = f"{arquivo} {sufixo_arquivo}"
    # Filtra conforme parâmetros obrigatórios
    df_filtrado = df.filter(
        (df.data >= data_minima) &
        (df.org_vendas == org_vendas)
    )
    # Filtros opcionais
    if canal_dist is not None:
        df_filtrado = df_filtrado.filter(df_filtrado.canal_dist == canal_dist)
    if centro_original is not None:
        df_filtrado = df_filtrado.filter(df_filtrado.centro_original == centro_original)
    if tipo_ov is not None:
        df_filtrado = df_filtrado.filter(df_filtrado.tipo_ov == tipo_ov)
    # Cria coluna de data no formato AAAA/MM
    df_formatado = df_filtrado.withColumn(
        "data_aaaa_mm", 
        date_format(df_filtrado.data, "yyyy/MM")
    )
    # Define colunas de agrupamento
    colunas_grupo = [linha]
    if incluir_cliente:
        colunas_grupo = ["emissor_da_ordem"] + colunas_grupo

    # Pivot
    if operacao == 'soma':
        df_pivot = df_formatado.groupBy(colunas_grupo).pivot("data_aaaa_mm").agg({'quantidade': 'sum'})
    elif operacao == 'contagem':
        df_pivot = df_formatado.groupBy(colunas_grupo).pivot("data_aaaa_mm").agg({'quantidade': 'count'})
    else:
        raise ValueError("Operação deve ser 'soma' ou 'contagem'")
    # Substitui nulos por zero
    df_pivot = df_pivot.fillna(0)
    # Adiciona colunas de arquivo e aba
    df_final = df_pivot.withColumn("file", lit(arquivo)).withColumn("sheet", lit(aba))
    # Renomeia colunas de agrupamento para nomes amigáveis
    nomes_amigaveis = {
        "item_principal_cadeia": "Item Principal Cadeia",
        "material": "Material",
        "emissor_da_ordem": "Cliente"
    }
    colunas_exibicao = []
    for col_grupo in colunas_grupo:
        nome_exibicao = nomes_amigaveis.get(col_grupo, col_grupo)
        df_final = df_final.withColumnRenamed(col_grupo, nome_exibicao)
        colunas_exibicao.append(nome_exibicao)
    # Reordena colunas
    cols = ["file", "sheet"] + colunas_exibicao + [col for col in df_final.columns if col not in ["file", "sheet"] + colunas_exibicao]
    df_final = df_final.select(*cols)
    return df_final

In [0]:
# Inicializa o DataFrame acumulador (vazio)
resultado_final = None

def append_resultado(novo_resultado):
    """Adiciona o resultado de gerar_layout_pivotado ao acumulador."""
    global resultado_final
    if resultado_final is None:
        resultado_final = novo_resultado
    else:
        resultado_final = resultado_final.unionByName(novo_resultado, allowMissingColumns=True)
    # Atualiza a temp view para consulta via SQL
    resultado_final.createOrReplaceTempView("vw_resultado_final")
    return resultado_final

In [0]:
# Parâmetros globais (widgets) — altere pela interface do notebook
dbutils.widgets.text("data_minima", "2025-01-01", "1 Data Mínima")
dbutils.widgets.text("ano", "2026", "2 Ano Referência")
dbutils.widgets.text("mes", "7", "3 Mês Referência")

data_minima = dbutils.widgets.get("data_minima")
mes = int(dbutils.widgets.get("mes"))
ano = int(dbutils.widgets.get("ano"))

# Mapeamento cadeia -> coluna de agrupamento
cadeia_map = {
    "fechada": "item_principal_cadeia",
    "aberta": "material"
}

# cliente: "aberto" = inclui emissor_da_ordem como agrupador, "fechado" = não inclui

# Configurações — só o que varia entre execuções
# "cadeia" define a coluna: "fechada" = Item Principal Cadeia, "aberta" = Material
configuracoes = [
    # DEMANDA FECHADA 2W
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    {
        "aba": "0203",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    {
        "aba": "0209",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    {
        "aba": "0232",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    # DEMANDA ABERTA 2W
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "aberta",
    },
    {
        "aba": "0203",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "aberta",
    },
    {
        "aba": "0209",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "aberta",
    },
    {
        "aba": "0232",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "aberta",
    },
    # DEMANDA LINHA 2W
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "operacao": "contagem",
        "sufixo_arquivo": "Demanda Linha",
        "cadeia": "fechada",
    },
    {
        "aba": "0203",
        "org_vendas": "0200",
        "operacao": "contagem",
        "sufixo_arquivo": "Demanda Linha",
        "cadeia": "fechada",
    },
    {
        "aba": "0209",
        "org_vendas": "0200",
        "operacao": "contagem",
        "sufixo_arquivo": "Demanda Linha",
        "cadeia": "fechada",
    },
    {
        "aba": "0232",
        "org_vendas": "0200",
        "operacao": "contagem",
        "sufixo_arquivo": "Demanda Linha",
        "cadeia": "fechada",
    },
    # DEMANDA MI 2W
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "canal_dist": "01",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    {
        "aba": "0203",
        "org_vendas": "0200",
        "canal_dist": "01",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    {
        "aba": "0209",
        "org_vendas": "0200",
        "canal_dist": "01",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    {
        "aba": "0232",
        "org_vendas": "0200",
        "canal_dist": "01",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    # DEMANDA MI 2W
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "canal_dist": "01",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    # DEMANDA ME
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "canal_dist": "02",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
    },
    # PEDIDO ZPUG
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
        "pedido": "ZPUG",
    },
    # PEDIDO ZPUG/CLIENTE
    {
        "aba": "TTL",
        "org_vendas": "0200",
        "operacao": "soma",
        "sufixo_arquivo": "Demanda Fechada",
        "cadeia": "fechada",
        "cliente": "aberto",
        "pedido": "ZPUG",
    },
]

# Executa todas as configurações com os parâmetros globais
for config in configuracoes:
    params = {k: v for k, v in config.items() if k not in ["cadeia", "cliente", "pedido"]}
    params["linha"] = cadeia_map[config["cadeia"]]
    params["incluir_cliente"] = config.get("cliente") == "aberto"
    # Mapeia "pedido" para tipo_ov
    if "pedido" in config:
        params["tipo_ov"] = config["pedido"]
    append_resultado(
        gerar_layout_pivotado(
            df=df,
            data_minima=data_minima,
            mes=mes,
            ano=ano,
            **params
        )
    )

print(f"{len(configuracoes)} execução(ões) acumuladas com sucesso.")
display(resultado_final)

In [0]:
# Grava o resultado acumulado na tabela Unity Catalog
# Usa column mapping para suportar espaços nos nomes de coluna
resultado_final.createOrReplaceTempView("_tmp_resultado_write")
spark.sql("""
  CREATE OR REPLACE TABLE parts_hdbk_sandbox.pr_demand.refined_demand_fechamento
  TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
  AS SELECT * FROM _tmp_resultado_write
""")

print("Tabela parts_hdbk_sandbox.pr_demand.refined_demand_fechamento gravada com sucesso.")